# E014B — fusion latente FreeQR, canal et timestep contrôlés

Ce notebook part du **blueprint exact-payload sélectionné par E014A**. Il ne prétend pas exécuter
un dépôt FreeQR officiel : il reconstruit et journalise le mécanisme publié de fusion d'une
représentation QR dans un canal latent, puis ajoute séparément une petite loss différentiable de
lecture. L'ablation est factorisée afin de savoir *ce qui* aide :

1. baseline sans fusion ;
2. canal latent 0, 1, 2 ou 3 ;
3. fenêtre temporelle early/middle/late/all ;
4. coefficient de fusion ;
5. meilleur réglage avec ou sans gradient de lecture.

Chaque exécution réutilise le même Stage 1, le même bruit initial et les mêmes paramètres
DiffQRCoder. Les frames montrent l'état latent décodé après chaque pas.


## Chaîne expérimentale

```text
latent Stage 2 z_t ─────────────────────────────────────────────────► z_0
       │              à chaque timestep sélectionné
       ├── canal c ← (1-α) canal c + α·QR_latent_bruité(t suivant)
       │
       └── option : gradient d'une loss de marge par module, canal c seulement

phase 1 : canal  →  phase 2 : fenêtre  →  phase 3 : α  →  phase 4 : gradient
             sélection scannabilité stricte avant toute esthétique
```

La fusion au callback a lieu **après** le pas DDIM ; la cible est donc bruitée au timestep suivant.
Ce choix est écrit dans chaque trace pour éviter l'erreur d'alignement d'un timestep.


In [ ]:
from __future__ import annotations

import gc
import json
import shutil
import sys
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import lpips
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from diffusers import ControlNetModel, DDIMScheduler
from IPython.display import Markdown, clear_output, display
from PIL import Image
from safetensors.torch import load_file, save_file

UPSTREAM_ROOT = Path('/opt/DiffQRCoder')
sys.path.insert(0, str(UPSTREAM_ROOT))
from diffqrcoder import DiffQRCoderPipeline  # noqa: E402
import diffqrcoder.srpg as upstream_srpg  # noqa: E402
from prooftag_qr.geometry import AlignedQR, aligned_module_diagnostics  # noqa: E402
from prooftag_qr.quality_scoring import CLIPQualityScorer  # noqa: E402
from prooftag_qr.validation import QRValidator, summarize_validation_records  # noqa: E402


class PaperLPIPSLoss(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.model = lpips.LPIPS(net='vgg', verbose=False)
        self.model.requires_grad_(False).eval()

    def forward(self, x, y):
        return self.model(x * 2 - 1, y * 2 - 1).mean()


upstream_srpg.PerceptualLoss = PaperLPIPSLoss
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))
free_gib, total_gib = torch.cuda.mem_get_info()
free_gib /= 2**30
total_gib /= 2**30
print(f'VRAM avant modèle : {free_gib:.2f} / {total_gib:.2f} Gio libres')
if free_gib < 18.0:
    raise RuntimeError(
        f'Seulement {free_gib:.2f} Gio libres avant le modèle. '
        'Un ancien kernel Jupyter utilise encore la RTX. Depuis PowerShell : '
        r'.\scripts\notebook-remote.ps1 -Reset '
        r'-Notebook 12_e014b_freeqr_latent_fusion.ipynb'
    )


## 1. Reprendre un résultat E014A sans deviner sa géométrie

In [ ]:
EXPERIMENT_NAME = 'e014b-freeqr-latent-channel-timestep-v1'
E014A_RUN_DIR = None  # Path('/data/notebook-runs/...-e014a-real-qart-exact-adaptive-v1')
PROMPT_ID = 'p1_simple'  # répéter ensuite p2/p3/p4 pour la confirmation
RESUME_RUN_NAME = None

if E014A_RUN_DIR is None:
    candidates = sorted([
        *Path('/data/notebook-runs').glob('*-e014a-deterministic-blueprint-pairing-v2'),
        *Path('/data/notebook-runs').glob('*-e014a-real-qart-exact-adaptive-v1'),
    ])
    if not candidates:
        raise FileNotFoundError('Aucun E014A : exécuter le notebook 11 avant E014B.')
    E014A_RUN_DIR = candidates[-1]
else:
    E014A_RUN_DIR = Path(E014A_RUN_DIR)
source_dir = E014A_RUN_DIR / PROMPT_ID
meta = json.loads((source_dir / 'selected-meta.json').read_text(encoding='utf-8'))
prompt = meta['prompt']
payload = meta['payload']
blueprint_image = Image.open(source_dir / 'selected-blueprint.png').convert('RGB')
matrix = np.load(source_dir / 'selected-matrix.npy').astype(np.uint8)
aligned = AlignedQR(
    image=blueprint_image, core_matrix=matrix, version=meta['version'],
    error_correction='M', mask_pattern=-1, module_size=meta['module_size'],
    padding_px=meta['padding_px'], canvas_size=meta['canvas_size'], payload=payload,
)
stage1_tensor = load_file(str(source_dir / 'stage1.safetensors'), device='cpu')['stage1']
stage1_image = Image.open(source_dir / 'stage1-reference.png').convert('RGB')

if RESUME_RUN_NAME:
    RUN_DIR = Path('/data/notebook-runs') / RESUME_RUN_NAME
else:
    RUN_DIR = Path('/data/notebook-runs') / (
        datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + EXPERIMENT_NAME
    )
    RUN_DIR.mkdir(parents=True)
RESULTS_PATH = RUN_DIR / 'results.jsonl'
print('Source :', E014A_RUN_DIR)
print('Blueprint exact :', meta['selected_blueprint'])
print('Sortie :', RUN_DIR)
display(blueprint_image.resize((430, 430)))


## 2. Paramètres fixes et plan d'ablation séquentiel

In [ ]:
BASE_MODEL_URL = 'https://huggingface.co/fp16-guy/Cetus-Mix_Whalefall_fp16_cleaned/blob/main/cetusMix_Whalefall2_fp16.safetensors'
CONTROLNET_MODEL = 'monster-labs/control_v1p_sd15_qrcode_monster'
CONTROLNET_SUBFOLDER = 'v2'
STEPS = 40
GUIDANCE_SCALE = 7.5
CONTROLNET_SCALE = 1.35
SCANNING_GUIDANCE = 500.0
PERCEPTUAL_GUIDANCE = 3.0
NEGATIVE_PROMPT = 'easynegative, unreadable text, letters, watermark'
SEED = next(item['seed'] for item in json.loads(
    (E014A_RUN_DIR / 'manifest.json').read_text(encoding='utf-8')
)['prompts'] if item['id'] == PROMPT_ID)
DISPLAY_EVERY = 5
SAVE_EVERY_STEP = True

BASE_CONFIG = {
    'channel': None, 'alpha': 0.0, 'window': [0.0, 1.0],
    'scan_gradient': False, 'scan_lr': 0.0, 'scan_every': 4,
}
CHANNEL_ALPHA = 0.15
WINDOWS = {
    'early': [0.00, 0.35], 'middle': [0.30, 0.70],
    'late': [0.65, 1.00], 'all': [0.00, 1.00],
}
ALPHAS = [0.05, 0.10, 0.15, 0.22]


## 3. Pipeline, latent QR propre et même bruit pour toutes les branches

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_MODEL, subfolder=CONTROLNET_SUBFOLDER, torch_dtype=torch.float16,
    cache_dir='/cache/huggingface',
)
pipe = DiffQRCoderPipeline.from_single_file(
    BASE_MODEL_URL, controlnet=controlnet, torch_dtype=torch.float16,
    cache_dir='/cache/huggingface', safety_checker=None, use_safetensors=True,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
for component in [pipe.unet, pipe.controlnet, pipe.vae, pipe.text_encoder]:
    component.requires_grad_(False).eval()
pipe.enable_attention_slicing('max')
pipe.enable_vae_slicing()
pipe.unet.enable_gradient_checkpointing()
pipe.controlnet.enable_gradient_checkpointing()
stage1_tensor = stage1_tensor.to('cuda', dtype=torch.float16)


@torch.no_grad()
def encode_image(image):
    array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to('cuda', torch.float16)
    return pipe.vae.encode(tensor * 2 - 1).latent_dist.mode() * pipe.vae.config.scaling_factor


@torch.no_grad()
def decode_latent(latent):
    decoded = pipe.vae.decode(
        latent.detach().to(dtype=next(pipe.vae.parameters()).dtype) / pipe.vae.config.scaling_factor,
        return_dict=False,
    )[0]
    return pipe.image_processor.postprocess(decoded.detach(), output_type='pil')[0].convert('RGB')


@torch.no_grad()
def initial_latent():
    encoded_reference = pipe.vae.encode(
        stage1_tensor * 2 - 1
    ).latent_dist.mode() * pipe.vae.config.scaling_factor
    generator = torch.Generator(device='cuda').manual_seed(SEED + 10000)
    noise = torch.randn(
        encoded_reference.shape, generator=generator, device='cuda',
        dtype=encoded_reference.dtype,
    )
    pipe.scheduler.set_timesteps(STEPS, device='cuda')
    return pipe.scheduler.add_noise(
        encoded_reference, noise, pipe.scheduler.timesteps[:1]
    ), noise


blueprint_latent = encode_image(blueprint_image)
paired_initial, paired_noise = initial_latent()
print('Latent image :', tuple(paired_initial.shape), 'latent blueprint :', tuple(blueprint_latent.shape))
assert paired_initial.shape[1] == 4


## 4. Callback de fusion et loss différentiable

La loss n'est **pas** le SRL officiel de DiffQRCoder : c'est une marge centrale simple, utilisée
uniquement pour tester l'interaction avec la fusion FreeQR. Son effet est séparé dans la phase 4.
Le latent rendu au pipeline est toujours détaché, ce qui évite l'erreur NumPy sur tensor avec grad.


In [ ]:
target_dark = torch.as_tensor(
    aligned.core_matrix.astype(bool), device='cuda'
).unsqueeze(0).unsqueeze(0)


def differentiable_module_loss(latent):
    dtype = next(pipe.vae.parameters()).dtype
    decoded = pipe.vae.decode(
        latent.to(dtype=dtype) / pipe.vae.config.scaling_factor, return_dict=False
    )[0]
    unit = (decoded.float() / 2 + 0.5).clamp(0, 1)
    gray = 0.299 * unit[:, 0:1] + 0.587 * unit[:, 1:2] + 0.114 * unit[:, 2:3]
    p = aligned.padding_px
    core = gray[:, :, p:p + aligned.core_size, p:p + aligned.core_size]
    modules = core.reshape(
        1, 1, aligned.core_modules, aligned.module_size,
        aligned.core_modules, aligned.module_size,
    ).permute(0, 1, 2, 4, 3, 5)
    inset = aligned.module_size // 3
    centers = modules[..., inset:-inset, inset:-inset].mean(dim=(-1, -2))
    dark_loss = F.relu(centers - 0.45)
    light_loss = F.relu(0.65 - centers)
    return torch.where(target_dark, dark_loss, light_loss).mean()


def fusion_active(step_index, window):
    progress = step_index / max(STEPS - 1, 1)
    return window[0] <= progress <= window[1]


def callback_for(config, output_dir):
    frames = output_dir / 'frames'
    frames.mkdir(parents=True, exist_ok=True)
    trace = []
    started = time.perf_counter()

    def callback(pipeline, step_index, timestep, callback_kwargs):
        latent = callback_kwargs['latents'].detach()
        next_timestep = (
            pipeline.scheduler.timesteps[step_index + 1]
            if step_index + 1 < len(pipeline.scheduler.timesteps)
            else torch.tensor(
                0, device=latent.device, dtype=pipeline.scheduler.timesteps.dtype
            )
        )
        fusion_applied = config['channel'] is not None and fusion_active(step_index, config['window'])
        scan_loss_value = None
        if fusion_applied:
            with torch.no_grad():
                noised_qr = pipeline.scheduler.add_noise(
                    blueprint_latent, paired_noise, next_timestep.reshape(1)
                )
                channel = config['channel']
                latent[:, channel:channel + 1] = (
                    (1 - config['alpha']) * latent[:, channel:channel + 1]
                    + config['alpha'] * noised_qr[:, channel:channel + 1]
                )
        if (
            config['scan_gradient'] and fusion_applied
            and step_index % config['scan_every'] == 0
        ):
            with torch.enable_grad():
                working = latent.detach().float().requires_grad_(True)
                loss = differentiable_module_loss(working)
                gradient = torch.autograd.grad(loss, working)[0]
                channel = config['channel']
                channel_gradient = gradient[:, channel:channel + 1]
                rms = channel_gradient.square().mean().sqrt().clamp_min(1e-8)
                updated = working.detach()
                updated[:, channel:channel + 1] -= (
                    config['scan_lr'] * channel_gradient / rms
                )
                latent = updated.to(dtype=callback_kwargs['latents'].dtype).detach()
                scan_loss_value = float(loss.detach().cpu())
        preview = decode_latent(latent)
        diagnostics = aligned_module_diagnostics(preview, aligned)
        row = {
            'step': int(step_index), 'timestep_before_step': int(timestep),
            'target_timestep_after_step': int(next_timestep),
            'fusion_applied': fusion_applied, 'scan_loss': scan_loss_value,
            'elapsed_s': time.perf_counter() - started, **diagnostics,
        }
        trace.append(row)
        if SAVE_EVERY_STEP or step_index % DISPLAY_EVERY == 0 or step_index == STEPS - 1:
            preview.save(frames / f'{step_index:03d}.jpg', quality=88)
        if step_index % DISPLAY_EVERY == 0 or step_index == STEPS - 1:
            clear_output(wait=True)
            display(Markdown(
                f"**{config['name']} — {step_index + 1}/{STEPS} — "
                f"fusion={fusion_applied} — MER={diagnostics['module_error_rate']:.2%}**"
            ))
            display(preview.resize((430, 430)))
        callback_kwargs['latents'] = latent.detach()
        return callback_kwargs

    return callback, trace


## 5. Exécuteur apparié, validation et persistance immédiate

In [ ]:
validator = QRValidator()
quality_scorer = CLIPQualityScorer(Path('/cache'), device='cpu')


def validation_summary(image):
    records = validator.validate(image, payload)
    summary = summarize_validation_records(records)
    passed = sum(item.exact_payload_match for item in records)
    return {
        'passed': passed, 'total': len(records), 'pass_rate': passed / len(records),
        'strict_all': passed == len(records),
        'worst_decoder_pass_rate': summary['worst_decoder_pass_rate'],
        'worst_scenario_pass_rate': summary['worst_scenario_pass_rate'],
    }, [asdict(item) for item in records]


def append_row(row):
    with RESULTS_PATH.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        stream.flush()


def completed_names():
    if not RESULTS_PATH.exists():
        return set()
    names = {
        json.loads(line)['name']
        for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()
    }
    for name in names:
        required = [
            RUN_DIR / name / 'final.png', RUN_DIR / name / 'final.safetensors',
            RUN_DIR / name / 'trace.json', RUN_DIR / name / 'validations.json',
            RUN_DIR / name / 'diffusion.gif',
        ]
        if not all(path.exists() for path in required):
            raise RuntimeError(
                f'Résultat {name} indexé mais artefacts incomplets ; '
                'restaurer le dossier avant la reprise.'
            )
    return names


def make_gif(folder, output):
    paths = sorted(folder.glob('*.jpg'))
    frames = [Image.open(path).convert('RGB').resize((512, 512)) for path in paths]
    if frames:
        frames[0].save(output, save_all=True, append_images=frames[1:], duration=150, loop=0)
    for frame in frames:
        frame.close()


def release_stage2_guidance():
    guidance = getattr(pipe, 'srpg', None)
    if guidance is not None:
        try:
            guidance.to('cpu')
        except Exception:
            pass
        pipe.srpg = None
        del guidance
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def run_config(config):
    if config['name'] in completed_names():
        print('SKIP', config['name'])
        return
    output_dir = RUN_DIR / config['name']
    output_dir.mkdir(parents=True, exist_ok=True)
    release_stage2_guidance()
    callback, trace = callback_for(config, output_dir)
    started = time.perf_counter()
    result = pipe._run_stage2(
        prompt=prompt, qrcode=blueprint_image,
        qrcode_module_size=aligned.module_size, qrcode_padding=aligned.padding_px,
        ref_image=stage1_tensor, negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE_SCALE, eta=0.0,
        generator=torch.Generator(device='cuda').manual_seed(SEED + 10000),
        latents=paired_initial.clone(), controlnet_conditioning_scale=CONTROLNET_SCALE,
        scanning_robust_guidance_scale=SCANNING_GUIDANCE,
        perceptual_guidance_scale=PERCEPTUAL_GUIDANCE,
        callback_on_step_end=callback, callback_on_step_end_tensor_inputs=['latents'],
        output_type='latent',
    )
    final_latent = result.images.detach()
    image = decode_latent(final_latent)
    duration = time.perf_counter() - started
    image.save(output_dir / 'final.png')
    save_file({'latents': final_latent.cpu().contiguous()}, str(output_dir / 'final.safetensors'))
    (output_dir / 'trace.json').write_text(json.dumps(trace, indent=2), encoding='utf-8')
    make_gif(output_dir / 'frames', output_dir / 'diffusion.gif')
    validation, records = validation_summary(image)
    (output_dir / 'validations.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
    try:
        quality = asdict(quality_scorer.score(image, prompt))
        quality_error = None
    except Exception as exc:
        quality = {'clip_similarity': None, 'clip_score': None, 'clip_aesthetic': None}
        quality_error = f'{type(exc).__name__}: {exc}'
    row = {
        'name': config['name'], 'prompt_id': PROMPT_ID, 'prompt': prompt,
        'duration_s': duration, 'config': config, **validation, **quality,
        **aligned_module_diagnostics(image, aligned), 'quality_error': quality_error,
    }
    append_row(row)
    print(config['name'], validation['passed'], '/', validation['total'])
    del result, final_latent
    release_stage2_guidance()
    gc.collect()
    torch.cuda.empty_cache()


def rows():
    return [
        json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ] if RESULTS_PATH.exists() else []


def rank(row):
    return (
        row['strict_all'], row['pass_rate'], row['worst_decoder_pass_rate'],
        row['worst_scenario_pass_rate'], row.get('clip_aesthetic') or -999,
        row.get('clip_score') or -999, -row['module_error_rate'],
    )


## 6. Phase 1 — trouver le canal

In [ ]:
run_config({'name': 'baseline_no_fusion', **BASE_CONFIG})
for channel in range(4):
    run_config({
        'name': f'channel_{channel}', **BASE_CONFIG,
        'channel': channel, 'alpha': CHANNEL_ALPHA, 'window': WINDOWS['all'],
    })
phase1 = [row for row in rows() if row['name'].startswith('channel_')]
best_channel_row = max(phase1, key=rank)
BEST_CHANNEL = best_channel_row['config']['channel']
print('Canal promu :', BEST_CHANNEL, best_channel_row['passed'], '/', best_channel_row['total'])
display(pd.DataFrame(phase1)[
    ['name', 'passed', 'total', 'module_error_rate', 'clip_aesthetic', 'clip_score']
])


## 7. Phase 2 — trouver la fenêtre temporelle

In [ ]:
for window_name, window in WINDOWS.items():
    run_config({
        'name': f'window_{window_name}', **BASE_CONFIG,
        'channel': BEST_CHANNEL, 'alpha': CHANNEL_ALPHA, 'window': window,
    })
phase2 = [row for row in rows() if row['name'].startswith('window_')]
best_window_row = max(phase2, key=rank)
BEST_WINDOW = best_window_row['config']['window']
print('Fenêtre promue :', BEST_WINDOW)
display(pd.DataFrame(phase2)[
    ['name', 'passed', 'total', 'module_error_rate', 'clip_aesthetic', 'clip_score']
])


## 8. Phase 3 — trouver la force de fusion

In [ ]:
for alpha in ALPHAS:
    run_config({
        'name': f'alpha_{alpha:.2f}', **BASE_CONFIG,
        'channel': BEST_CHANNEL, 'alpha': alpha, 'window': BEST_WINDOW,
    })
phase3 = [row for row in rows() if row['name'].startswith('alpha_')]
best_alpha_row = max(phase3, key=rank)
BEST_ALPHA = best_alpha_row['config']['alpha']
print('Alpha promu :', BEST_ALPHA)
display(pd.DataFrame(phase3)[
    ['name', 'passed', 'total', 'module_error_rate', 'clip_aesthetic', 'clip_score']
])


## 9. Phase 4 — isoler l'apport du gradient de lecture

In [ ]:
for scan_lr in [0.01, 0.03, 0.06]:
    run_config({
        'name': f'best_gradient_{scan_lr:.2f}', **BASE_CONFIG,
        'channel': BEST_CHANNEL, 'alpha': BEST_ALPHA, 'window': BEST_WINDOW,
        'scan_gradient': True, 'scan_lr': scan_lr, 'scan_every': 4,
    })
all_rows = rows()
winner = max(all_rows, key=rank)
pd.DataFrame(all_rows).to_csv(RUN_DIR / 'comparison.csv', index=False)
display(pd.DataFrame(all_rows).sort_values(
    ['strict_all', 'pass_rate', 'clip_aesthetic'], ascending=False
)[['name', 'passed', 'total', 'clip_aesthetic', 'clip_score', 'module_error_rate', 'duration_s']])
print('Gagnant observé :', winner['name'])
if not winner['strict_all']:
    print('NON LIVRABLE : aucune configuration n a franchi tous les tests logiciels.')


## 10. Courbes, manifeste et archive

In [ ]:
winner_trace = json.loads(
    (RUN_DIR / winner['name'] / 'trace.json').read_text(encoding='utf-8')
)
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot([row['step'] for row in winner_trace], [row['module_error_rate'] for row in winner_trace])
axes[0].set(title='MER par pas', xlabel='pas', ylabel='MER')
axes[1].plot(
    [row['step'] for row in winner_trace],
    [np.nan if row['scan_loss'] is None else row['scan_loss'] for row in winner_trace],
)
axes[1].set(title='Loss différentiable (seulement aux pas actifs)', xlabel='pas', ylabel='loss')
for axis in axes:
    axis.grid(alpha=0.25)
figure.tight_layout()
figure.savefig(RUN_DIR / 'winner-trace.png', dpi=160)
display(figure)

manifest = {
    'experiment': EXPERIMENT_NAME, 'source_e014a': str(E014A_RUN_DIR),
    'prompt_id': PROMPT_ID, 'seed': SEED, 'payload': payload,
    'selected_e014a_blueprint': meta['selected_blueprint'],
    'steps': STEPS, 'winner': winner['name'], 'winner_strict': winner['strict_all'],
    'best_channel': BEST_CHANNEL, 'best_window': BEST_WINDOW, 'best_alpha': BEST_ALPHA,
    'timestep_alignment': 'callback after scheduler step uses next scheduler timestep',
    'claim': 'FreeQR-inspired channel/timestep reconstruction, not an official FreeQR code release',
    'gradient_claim': 'Prooftag central-module margin loss, not DiffQRCoder SR-MPGD',
}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(RUN_DIR), 'gztar', RUN_DIR.parent, RUN_DIR.name)
print('Archive :', archive)
